Downloading from this list may bring yml files that does not exist in the repository anymore, so it will be misleading for my study.

In [5]:
import os
import requests
import pandas as pd
from urllib.parse import urlparse
from time import sleep
from dotenv import load_dotenv, dotenv_values

# === Load GitHub tokens from .env file ===
env_path = r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env"
load_dotenv(env_path)
env_vars = dotenv_values(env_path)

# Load all available GitHub tokens
GITHUB_TOKENS = [env_vars.get(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if env_vars.get(f"GITHUB_TOKEN_{i}")]
if not GITHUB_TOKENS:
    raise Exception("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0

# === CONFIGURATION ===
base_folder = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Download_Only_YMLs_from_Step4"
input_csv = os.path.join(base_folder, "step4_detected_yml_files.csv")
output_csv = os.path.join(base_folder, "step4_download_log.csv")
download_root = os.path.join(base_folder, "Downloaded_YMLs")

# === Load CSV ===
df = pd.read_csv(input_csv)
df['download_status'] = ""
df['download_reason'] = ""

# === Ensure output folder exists ===
os.makedirs(download_root, exist_ok=True)

# === Loop through each file ===
for idx, row in df.iterrows():
    html_url = row['html_url']
    file_path = row['file_path']
    parsed = urlparse(html_url)
    path_parts = parsed.path.strip("/").split("/")
    if len(path_parts) < 2:
        df.at[idx, 'download_status'] = "failed"
        df.at[idx, 'download_reason'] = "Invalid URL"
        continue

    owner, repo = path_parts[0], path_parts[1]
    repo_name = f"{owner}_{repo}"
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{file_path}"

    headers = {
        "Accept": "application/vnd.github.v3.raw",
        "Authorization": f"Bearer {GITHUB_TOKENS[token_index]}"
    }

    try:
        response = requests.get(api_url, headers=headers)
        if response.status_code == 403:
            # Rotate token
            token_index = (token_index + 1) % len(GITHUB_TOKENS)
            headers["Authorization"] = f"Bearer {GITHUB_TOKENS[token_index]}"
            response = requests.get(api_url, headers=headers)

        if response.status_code == 200:
            # Save file
            file_name = os.path.basename(file_path)
            save_dir = os.path.join(download_root, repo_name)
            os.makedirs(save_dir, exist_ok=True)
            with open(os.path.join(save_dir, file_name), 'wb') as f:
                f.write(response.content)
            df.at[idx, 'download_status'] = "success"
        else:
            df.at[idx, 'download_status'] = "failed"
            df.at[idx, 'download_reason'] = f"HTTP {response.status_code}"
    except Exception as e:
        df.at[idx, 'download_status'] = "failed"
        df.at[idx, 'download_reason'] = str(e)

    print(f"[{idx+1}/{len(df)}] {file_path} => {df.at[idx, 'download_status']}")

    sleep(0.5)  # avoid rate limits

# === Save log ===
df.to_csv(output_csv, index=False)
print(f"\n✅ Log saved to: {output_csv}")


[1/11567] .github/workflows/ci.yml => success
[2/11567] .github/workflows/export-po-files.yml => success
[3/11567] .github/workflows/gradle-wrapper-validation.yml => success
[4/11567] .github/workflows/translations-import.yml => success
[5/11567] .travis.yml => success
[6/11567] .github/workflows/build-container.yml => success
[7/11567] .github/workflows/build-docs.yml => success
[8/11567] .github/workflows/build-kernel.yml => success
[9/11567] .github/workflows/build-native.yml => success
[10/11567] .github/workflows/build-translation.yml => success
[11/11567] .gitlab-ci.yml => success
[12/11567] .github/workflows/android.yml => success
[13/11567] .github/workflows/build.yml => success
[14/11567] .github/workflows/merge.yml => success
[15/11567] .github/workflows/post_build.yml => success
[16/11567] .github/workflows/post_merge.yml => success
[17/11567] .travis.yml => success
[18/11567] .travis.yml => success
[19/11567] .travis.yml => success
[20/11567] .travis.yml => success
[21/1156